# Notebook 8d: Is the markdown bias only an E5 problem? (RQ5, more models)

**Before running:** choose a **T4 GPU** (Runtime → Change runtime type). About **25–35 minutes**. Notebooks 8b and 8c must have run first. If Colab disconnects, just run all cells again: every finished model is saved on Drive and skipped.

**Why this notebook.** Notebook 8c showed that adding markdown to the original resumes, with no word changed, raises E5 by +0.91 SD. BGE rose only +0.18 and MPNet not at all. A reviewer will ask: *is this one model's quirk, or do embedding models in general do this?* Here we repeat the 8c test on **eight models**: E5-base again (as a check that we reproduce 8c), plus seven others that are widely used, including one larger model of the same family, a multilingual model, a very small model, a long-context model and a recent LLM-based embedder.

**Same versions as 8c** (built with the same code; letters and digits never change): `flat`, `lines`, `bold`, `bullets`, `full`. Every other candidate keeps their **original** resume, so "places up" means the same as in 8c.

**Decision rule, fixed before running (2026-10-11).** Only the **seven new models** count (E5-base is the reproduction check). A model *shows the bias* if its **all formatting** step (flat → full) is at least **+0.30 SD** with **Holm p < 0.05** across the seven.
- **5 or more of 7** show it → the paper says *embedding models commonly reward markdown formatting*.
- **2 to 4 of 7** → *several embedding models, most strongly E5*, and we name them.
- **0 or 1 of 7** → *E5 (and to a lesser degree BGE)*; the paper presents it as a model-specific risk that should be checked per model.

Whatever happens, every model's number goes into the paper's table.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers einops

import os, re, gc
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('running on', DEVICE)
assert DEVICE == 'cuda', 'No GPU. Runtime > Change runtime type > T4 GPU, then run again (CPU would take hours).'
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/08d_cache'
os.makedirs(CACHE, exist_ok=True)
SEED = 42
MAX_WORDS = 900

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index
evald = all_df[all_df.job_id.isin(usable)].reset_index(drop=True)
evald['pair_key'] = evald.job_id + '|' + evald.resume_id

def cut(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words])
evald['resume_cut'] = evald.resume_text.map(lambda t: cut(t, MAX_WORDS))

nb8c = pd.read_parquet(f'{RESULTS}/08c_pair_scores.parquet')
ids = sorted(nb8c.resume_id.unique())
print(f'{len(usable)} usable jobs, {len(evald)} pairs, {len(ids)} resumes from Notebook 8c')
assert len(ids) == 181, 'Not the same resumes as Notebook 8c.'

## 1. Build the versions (the same code as Notebook 8c)

In [ ]:
HEADINGS = ['Professional Summary', 'Executive Summary', 'Executive Profile', 'Career Overview', 'Professional Profile',
            'Professional Experience', 'Work History', 'Work Experience', 'Experience', 'Education and Training',
            'Education', 'Skill Highlights', 'Core Qualifications', 'Core Competencies', 'Qualifications', 'Skills',
            'Summary', 'Highlights', 'Accomplishments', 'Certifications', 'Additional Information', 'Languages',
            'Interests', 'Affiliations', 'Training', 'Projects', 'Publications', 'Awards', 'Objective']
HEAD = re.compile(r'(?<![A-Z])(' + '|'.join(HEADINGS) + r')(?=[A-Z0-9])')
H0, H1, BR = '\x01', '\x02', '\x00'   # heading start, heading end, item break

JOINED = re.compile(r'[a-z][A-Z]|[A-Z]{2}[a-z]{2}')
jobs_u = evald.drop_duplicates('job_id')
job_df = {}
for t in jobs_u.job_text:
    for w in set(re.findall(r'[A-Za-z]+', t)):
        if JOINED.search(w):
            job_df[w] = job_df.get(w, 0) + 1
KEEP = {w for w, n in job_df.items() if n >= 2}

def split_word(w):
    if w in KEEP or not JOINED.search(w):
        return w
    s = re.sub(r'^to(?=[A-Z])', 'to ', w)                        # 2017toCurrent is a date range, not an item break
    s = re.sub(r'([a-z])([A-Z])', r'\1' + BR + r'\2', s)        # a glued item boundary becomes an item break
    return re.sub(r'([A-Z]+)([A-Z][a-z]{2,})', r'\1' + BR + r'\2', s)

def structure(t):
    t = HEAD.sub(lambda m: H0 + m.group(1) + H1, t)
    t = re.sub(r'[A-Za-z]+', lambda m: split_word(m.group()), t)
    t = re.sub(r'(?<=[a-z]{4})(?=\d)', ' ', t)
    t = re.sub(r'(?<=\d)(?=to[A-Z ])|(?<=\d)(?=[A-Z][a-z])', ' ', t)
    t = re.sub(r'(?<=[a-z]{2}[.!?]) ?(?=[A-Z])', BR, t)            # sentence end: an item break
    t = re.sub(r'(?<=[a-z]{2}[;:])(?=[A-Z])', ' ', t)
    t = re.sub(r'(?<=[A-Za-z]{2}[,;])(?=[A-Za-z])', ' ', t)
    t = re.sub(r'\s*\|\s*', ' | ', t)
    return t

def render(t, style):
    head_on, head_off = ('**', ':**') if style in ('bold', 'full') else ('', ':')
    bullet = '* ' if style in ('bullets', 'full') else ''
    if style == 'flat':
        t = t.replace(H0, ' ').replace(H1, ' ').replace(BR, ' ')
        return re.sub(r'\s+', ' ', t).strip()
    t = t.replace(H0, '\n\n' + head_on).replace(H1, head_off + '\n').replace(BR, '\n')
    out = []
    for line in t.split('\n'):
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if not line:
            if out and out[-1] != '':
                out.append('')
            continue
        is_head = line.startswith('**') and line.endswith(':**') or (style in ('lines', 'bullets') and line.endswith(':') and line[:-1] in HEADINGS)
        out.append(line if is_head else bullet + line)
    return '\n'.join(out).strip()

STYLES = ['flat', 'lines', 'bold', 'bullets', 'full']
res_cut = evald.drop_duplicates('resume_id').set_index('resume_id').resume_cut
struct = {r: structure(t) for r, t in res_cut.items()}
V = {s: {r: render(t, s) for r, t in struct.items()} for s in STYLES}

same = lambda a, b: re.sub(r'\W', '', a) == re.sub(r'\W', '', b)
for s in STYLES:
    bad = [r for r in ids if not same(V[s][r], res_cut[r])]
    assert not bad, f'{s}: content changed for {len(bad)} resumes.'
print('check passed: every version has exactly the same letters and digits as the original')
n_head = pd.Series({r: t.count(H0) for r, t in struct.items()})[ids]
n_items = pd.Series({r: t.count(BR) for r, t in struct.items()})[ids]
print('headings found per resume:', n_head.describe().round(1).to_dict())
print('item breaks per resume:', n_items.describe().round(1).to_dict())
print('resumes with no heading found:', int((n_head == 0).sum()))
print('\n===== FULL version, first 40 lines of one resume =====')
print('\n'.join(V['full'][ids[0]].split('\n')[:40]))

## 2. Score every version with every model

Prefixes follow each model's own instructions (E5: `query:` / `passage:`; Nomic: `search_query:` / `search_document:`; Qwen3-Embedding: a one-line task instruction on the job side only). BGE and GTE get no prefix, the same as BGE-base in 8c. All inputs are cut at 512 tokens, as in 8c. If one model fails to load, the notebook prints why and moves on; which one.

In [ ]:
from sentence_transformers import SentenceTransformer

QWEN_Q = 'Instruct: Given a job description, retrieve resumes of candidates who fit the job\nQuery: '
MODELS = [  # (column, Hugging Face id, size, job prefix, resume prefix)
    ('e5',       'intfloat/e5-base-v2',                     '110M', 'query: ', 'passage: '),   # reproduction check
    ('e5large',  'intfloat/e5-large-v2',                    '335M', 'query: ', 'passage: '),
    ('me5',      'intfloat/multilingual-e5-base',           '278M', 'query: ', 'passage: '),
    ('bgelarge', 'BAAI/bge-large-en-v1.5',                  '335M', '', ''),
    ('gte',      'thenlper/gte-base',                       '110M', '', ''),
    ('minilm',   'sentence-transformers/all-MiniLM-L6-v2',  '23M',  '', ''),
    ('nomic',    'nomic-ai/nomic-embed-text-v1.5',          '137M', 'search_query: ', 'search_document: '),
    ('qwen3emb', 'Qwen/Qwen3-Embedding-0.6B',               '600M', QWEN_Q, ''),
]
SIZE = {c: s for c, _, s, _, _ in MODELS}
HF = {c: h for c, h, _, _, _ in MODELS}

res_ids = list(res_cut.index)
in_target = evald.resume_id.isin(ids)
failed = {}
for col, hf_id, _, qp, dp in MODELS:
    path = f'{CACHE}/{col}.parquet'
    if os.path.exists(path):
        print(col, 'already done (cached)'); continue
    try:
        kw = {'trust_remote_code': True} if col == 'nomic' else {}
        if col == 'qwen3emb':
            kw['model_kwargs'] = {'torch_dtype': torch.float16}
        m = SentenceTransformer(hf_id, device=DEVICE, **kw)
        m.max_seq_length = 512
        enc = lambda texts, p: m.encode([p + t for t in texts], batch_size=16, normalize_embeddings=True,
                                        show_progress_bar=False)
        je = dict(zip(jobs_u.job_id, enc(jobs_u.job_text.tolist(), qp)))
        out = evald[['pair_key', 'job_id', 'resume_id', 'relevance']].copy()
        texts = {'orig': [res_cut[r] for r in res_ids], 'allflat': [V['flat'][r] for r in res_ids],
                 'allfull': [V['full'][r] for r in res_ids]}
        for v, tx in texts.items():
            rv = dict(zip(res_ids, enc(tx, dp)))
            out[v] = [float(je[j] @ rv[r]) for j, r in zip(out.job_id, out.resume_id)]
        for s in ('lines', 'bold', 'bullets'):
            rv = dict(zip(ids, enc([V[s][r] for r in ids], dp)))
            out[s] = [float(je[j] @ rv[r]) if t else np.nan for j, r, t in zip(out.job_id, out.resume_id, in_target)]
        out.to_parquet(path, index=False)
        print(col, 'done', flush=True)
        del m
    except Exception as e:
        failed[col] = repr(e)[:300]
        print(col, 'FAILED:', failed[col], flush=True)
    gc.collect(); torch.cuda.empty_cache()

DONE = [c for c, *_ in MODELS if os.path.exists(f'{CACHE}/{c}.parquet')]
S = {c: pd.read_parquet(f'{CACHE}/{c}.parquet') for c in DONE}
print('models scored:', DONE, '| failed:', failed or 'none')

# Reproduction check: E5-base here must match Notebook 8c
ref = nb8c.set_index('pair_key')
e5t = S['e5'][S['e5'].resume_id.isin(ids)].set_index('pair_key')
for here, there in [('allflat', 'e5_flat'), ('allfull', 'e5_full'), ('bold', 'e5_bold'), ('orig', 'e5_orig')]:
    d = (e5t[here] - ref[there]).abs()
    print(f'E5 {here:8} vs 8c {there:8}: mean difference {d.mean():.5f}, max {d.max():.5f}')
print('(a mean difference above 0.01 means something changed since 8c)')

## 3. Results: what does each formatting step do, per model?

Same steps and statistics as Notebook 8c: change in SDs of each model's own score spread, cluster bootstrap CI by resume, Wilcoxon p, ranks among the job's original candidates. Holm correction is across the **seven new models** within each step (E5-base is shown but not counted).

In [ ]:
def holm(p):
    p = np.asarray(p, dtype=float); order = np.argsort(p); k = len(p); adj = np.empty(k); running = 0
    for rank, i in enumerate(order):
        running = max(running, (k - rank) * p[i]); adj[i] = min(1, running)
    return adj

def cluster_ci(d, groups, n=2000):
    df = pd.DataFrame({'d': d, 'g': groups}); sums = df.groupby('g').d.agg(['sum', 'count'])
    rng = np.random.default_rng(SEED); ids_ = np.arange(len(sums)); out = []
    for _ in range(n):
        b = sums.iloc[rng.choice(ids_, len(ids_))]
        out.append(b['sum'].sum() / b['count'].sum())
    return np.percentile(out, [2.5, 97.5])

STEPS = [('colons + line breaks', 'allflat', 'lines'), ('bold headings', 'lines', 'bold'),
         ('bullets', 'lines', 'bullets'), ('all formatting', 'allflat', 'allfull')]
rows = []
for col in DONE:
    df = S[col]
    sd = df.orig.std()
    by_job = df.groupby('job_id').orig.apply(np.array).to_dict()
    t = df[df.resume_id.isin(ids)].copy()
    rank = lambda v: np.array([1 + int((by_job[j] > x).sum()) - int(o > x) for j, o, x in zip(t.job_id, t.orig, t[v])])
    for step, a, b in STEPS:
        d = (t[b] - t[a]).to_numpy(); lo, hi = cluster_ci(d, t.resume_id.to_numpy()); ra, rb = rank(a), rank(b)
        rows.append({'model': col, 'size': SIZE[col], 'step': step, 'n': len(d),
                     'change in SDs': round(d.mean() / sd, 3), 'CI (SDs)': f'[{lo / sd:.3f}, {hi / sd:.3f}]',
                     'share up': round((d > 0).mean(), 3), 'mean places up': round((ra - rb).mean(), 2),
                     'share reaching top 5': round(((ra > 5) & (rb <= 5)).mean(), 3),
                     'p': wilcoxon(d).pvalue if np.any(d != 0) else 1.0})
steps = pd.DataFrame(rows)
new = steps.model != 'e5'
steps['p (Holm, 7 new)'] = np.nan
for step in steps.step.unique():
    k = new & (steps.step == step)
    steps.loc[k, 'p (Holm, 7 new)'] = holm(steps.loc[k, 'p'].to_numpy())
pd.set_option('display.width', 250)
print(steps.pivot(index='model', columns='step', values='change in SDs').loc[DONE, [s for s, _, _ in STEPS]], '\n')

# Fit minus no-fit (per resume), all formatting: is the boost fit-blind for every model?
gaps = []
for col in DONE:
    t = S[col][S[col].resume_id.isin(ids)]
    d = t.assign(d=t.allfull - t.allflat, fit=t.relevance > 0)
    per = d.groupby(['resume_id', 'fit']).d.mean().unstack().dropna()
    gap = (per[True] - per[False]) / S[col].orig.std()
    gaps.append({'model': col, 'fit minus no-fit (SDs)': round(gap.mean(), 3), 'resumes': len(gap),
                 'p': wilcoxon(gap).pvalue})
gaps = pd.DataFrame(gaps)
print(gaps.to_string(index=False), '\n')

# Decision rule (fixed before running, see the top of the notebook)
allf = steps[(steps.step == 'all formatting') & new]
shows = allf[(allf['change in SDs'] >= 0.30) & (allf['p (Holm, 7 new)'] < 0.05)]
n_new = len(allf)
print(f'{len(shows)} of {n_new} new models show the bias (>= +0.30 SD, Holm p < 0.05):', list(shows.model))
if n_new < 7:
    print('WARNING: fewer than 7 new models finished; before reading the rule.')
verdict = ('embedding models commonly reward markdown' if len(shows) >= 5 else
           'several embedding models, most strongly E5' if len(shows) >= 2 else
           'mainly E5 (model-specific risk)')
print('Paper wording per the rule:', verdict)
steps[steps.step == 'all formatting']

## 4. If every resume had markdown, would ranking change?

As in 8c section 5: all candidates `full` vs all `flat`, NDCG@10 per job, paired permutation test. Also each model's NDCG@10 on the **original** text, for context.

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k])
    return dcg(r[:k]) / ideal if ideal > 0 else 0.0
def paired_permutation_p(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed); d = np.asarray(a) - np.asarray(b)
    signs = rng.choice([-1, 1], size=(n, len(d)))
    return float((np.abs((signs * d).mean(axis=1)) >= abs(d.mean())).mean())

rows = []
for col in DONE:
    per = {}
    for v in ('orig', 'allflat', 'allfull'):
        rng = np.random.default_rng(SEED); vals = []
        for jid, grp in S[col].groupby('job_id', sort=True):
            s = grp[v].to_numpy(); rel = grp.relevance.to_numpy()
            vals.append(ndcg_at_k(rel[np.lexsort((rng.random(len(s)), -s))].tolist(), 10))
        per[v] = np.array(vals)
    rows.append({'model': col, 'jobs': len(per['orig']), 'NDCG@10 original': round(per['orig'].mean(), 4),
                 'NDCG@10 all flat': round(per['allflat'].mean(), 4), 'NDCG@10 all markdown': round(per['allfull'].mean(), 4),
                 'diff': round((per['allfull'] - per['allflat']).mean(), 4),
                 'p': paired_permutation_p(per['allfull'], per['allflat'])})
everyone = pd.DataFrame(rows)
everyone['p (Holm)'] = holm(everyone.p.to_numpy())
everyone

## 5. Save

In [ ]:
steps.assign(hf_id=steps.model.map(HF)).to_csv(f'{RESULTS}/08d_markdown_steps_models.csv', index=False)
gaps.to_csv(f'{RESULTS}/08d_fit_gap_models.csv', index=False)
everyone.to_csv(f'{RESULTS}/08d_everyone_ndcg_models.csv', index=False)
pd.DataFrame({'failed_model': list(failed), 'error': list(failed.values())}).to_csv(f'{RESULTS}/08d_failed.csv', index=False)
print('saved. Next: download this notebook (File > Download > .ipynb).')